In [1]:
import pandas as pd
import tqdm, os
import pickle
from sklearn.preprocessing import StandardScaler
import warnings
warnings.simplefilter("ignore")

import sys
sys.path.insert(0, "../../..")

from utils.information_theoretic_calculation import InformationTheoreticCalculator
from utils.ipa_processing import syllabify_ipa
SEP = "."

In [2]:
df = pd.read_csv("../dataset_construction/datasets/processed_aemri.csv")
df["subject"] = df["art_f"].apply(lambda x: os.path.basename(x).split("_")[2])

In [3]:
df

,syllable1_start,syllable1_end,syllable2_start,syllable2_end,syllable1,syllable2,context,word,word_id,art_f,ac_f,subject
0,2.432700,2.642700,2.642700,2.862700,ɔ,b_ɝ_n,ɔ,auburn,1.0,/Volumes/Transcend/USC-TIMIT/MRI/Data/M1/avi/u...,/Volumes/Transcend/USC-TIMIT/MRI/Data/M1/wav/u...,m1
1,3.272700,3.362700,3.362700,3.562700,ɹ_i,m_aɪ_n,ɹ_i,reminded,3.0,/Volumes/Transcend/USC-TIMIT/MRI/Data/M1/avi/u...,/Volumes/Transcend/USC-TIMIT/MRI/Data/M1/wav/u...,m1
2,3.362700,3.562700,3.562700,3.712700,m_aɪ_n,d_ɪ_d,ɹ_i.m_aɪ_n,reminded,3.0,/Volumes/Transcend/USC-TIMIT/MRI/Data/M1/avi/u...,/Volumes/Transcend/USC-TIMIT/MRI/Data/M1/wav/u...,m1
3,7.722700,7.882700,7.882700,8.212700,ɹ_ɪ,m_eɪ_n_d,ɹ_ɪ,remained,9.0,/Volumes/Transcend/USC-TIMIT/MRI/Data/M1/avi/u...,/Volumes/Transcend/USC-TIMIT/MRI/Data/M1/wav/u...,m1
4,8.212700,8.462700,8.462700,8.842700,l_aɪ,f_l_ɔ_ŋ,l_aɪ,lifelong,10.0,/Volumes/Transcend/USC-TIMIT/MRI/Data/M1/avi/u...,/Volumes/Transcend/USC-TIMIT/MRI/Data/M1/wav/u...,m1
...,...,...,...,...,...,...,...,...,...,...,...,...
18525,16.372356,16.592576,16.592576,16.922906,aɪ,d_ɹ_ɑ_p_s,aɪ,EYEDROPS,28.0,/Volumes/Transcend/USC-TIMIT/MRI/Data/F1/avi/u...,/Volumes/Transcend/USC-TIMIT/MRI/Data/F1/wav/u...,f1
18526,21.620750,21.710840,21.710840,21.800930,s_ɛ_l,d_ʌ_m,s_ɛ_l,SELDOM,33.0,/Volumes/Transcend/USC-TIMIT/MRI/Data/F1/avi/u...,/Volumes/Transcend/USC-TIMIT/MRI/Data/F1/wav/u...,f1
18527,21.800930,21.860990,21.860990,21.951080,w_ɑ,tʃ_ɪ_z,w_ɑ,WATCHES,34.0,/Volumes/Transcend/USC-TIMIT/MRI/Data/F1/avi/u...,/Volumes/Transcend/USC-TIMIT/MRI/Data/F1/wav/u...,f1
18528,21.951080,22.211340,22.211340,22.561690,d_eɪ,t_aɪ_m,d_eɪ,DAYTIME,35.0,/Volumes/Transcend/USC-TIMIT/MRI/Data/F1/avi/u...,/Volumes/Transcend/USC-TIMIT/MRI/Data/F1/wav/u...,f1


In [4]:
with open("../../../phone_based/pronunciation_counts/pronunciation_counts_ae.pkl", "rb") as f:
    pronunciation_counts = pickle.load(f)
pronunciation_counts = {syllabify_ipa(k, lang="ae"): v for k, v in pronunciation_counts.items() if "*" not in k and k != ""}

In [5]:
bigrams = set()
for syllable1, syllable2 in tqdm.tqdm(zip(df["syllable1"], df["syllable2"]), total=len(df)):
    bigrams.add(syllable1+SEP+syllable2)

all_phones = set()
for pronunciation in tqdm.tqdm(pronunciation_counts):

    for phone in pronunciation.split(SEP):
        all_phones.add(phone)

monogram_counts = {}
for pronunciation, count in tqdm.tqdm(pronunciation_counts.items()):
    syllables = pronunciation.split(SEP)
    for syllable in syllables:
        if syllable not in monogram_counts:
            monogram_counts[syllable] = 0
        monogram_counts[syllable] += count

100%|██████████| 48576/48576 [00:00<00:00, 1586050.89it/s]


In [6]:
calculator = InformationTheoreticCalculator(
    pronunciation_counts=pronunciation_counts,
    phone_counts=monogram_counts,
    sep=SEP
)

  0%|          | 0/48576 [00:00<?, ?it/s]

100%|██████████| 48576/48576 [00:00<00:00, 310977.51it/s]


In [7]:
df["bigram_surprisal"] = df.apply(lambda x: calculator.get_contextual_surprisal(SEP.join(x["context"].split(SEP)[:-1]), x["syllable1"] + SEP + x["syllable2"]), axis=1)
df["syllable2_pos_entropy"] = df.apply(lambda x: calculator.get_target_pos_entropy(x["context"]), axis=1)